## Parte 1. Conceptos

1. Que es aprendizaje supervisado?
Es crear un algoritmo capaz de predecir un resultado usando datos iniciales con respuesta
2. Que es una variable objetivo?
Es la variable a predecir
3. Que significa `venta_alta`?
Define si una venta es alta o no
4. Que diferencia hay entre `X` e `y`?
'x' son los datos de entrada y 'y' es la variable objetivo
5. Que es clasificacion?
Es clasificar una venta como venta alta o no alta
6. Por que este problema es de clasificacion?
Porque se intentara predecir si una venta es alta o no alta
7. Que hace `DecisionTreeClassifier`?
Entrena un modelo
8. Para que sirve separar entrenamiento y prueba?
Para poder entrenar el modelo con una cierta cantidad de datos y probar con la otra cantidad restante
9. Que significa exactitud?
Es la precisión del modelo (de si acerto o no)
10. Para que sirve una matriz de confusion?
Para saber las coincidencias correctas y las coincidencias incorrectas
11. Que significa `fit()`?
Es una función que sirve para entrenar el modelo
12. Que significa `predict()`?
Es una función que sirve para predecir los resultados
13. Por que se usa `pd.get_dummies()`?
Para convertir las columnas a datos numericos (estaban en texto)
14. Por que se usa `reindex()` al predecir ventas nuevas?
Reindex alinea las columnas nuevas con las columnas del entrenamiento, si esto no se hace los datos quedan disparejos y la predicción o saldria mal o fallaria
15. Por que una prediccion no es una verdad absoluta?
Porque en la realidad el modelo prodria fallar y no dar una predicción correcta

## Parte 2. Revision Del Dataset

In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

In [2]:
df = pd.read_csv("ventas_ecommerce_limpio.csv")

In [3]:
#1. Muestra las primeras filas.
df.head()

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,total_venta
0,1001,2026-07-01,Ana Lopez,Mouse,Accesorios,1,250.0,Efectivo,Cuernavaca,250.0
1,1002,2026-07-01,Luis Perez,Teclado,Accesorios,1,650.0,Tarjeta,Jiutepec,650.0
2,1003,2026-07-02,Sofia Ruiz,Audifonos,Accesorios,1,900.0,Tarjeta,Temixco,900.0
3,1004,2026-07-02,Pedro Mata,Webcam,Accesorios,1,800.0,Efectivo,Cuernavaca,800.0
4,1005,2026-07-03,Laura Diaz,Cable HDMI,Accesorios,2,180.0,Efectivo,Jiutepec,360.0


In [4]:
#2. Revisa las columnas.
df.columns

Index(['id_venta', 'fecha', 'cliente', 'producto', 'categoria', 'cantidad',
       'precio_unitario', 'metodo_pago', 'ciudad', 'total_venta'],
      dtype='object')

In [5]:
#3. Muestra cuantas filas y columnas tiene.
df.shape

(60, 10)

In [6]:
#4. Revisa si hay valores nulos.
df.isnull().sum()

id_venta           0
fecha              0
cliente            0
producto           0
categoria          0
cantidad           0
precio_unitario    0
metodo_pago        0
ciudad             0
total_venta        0
dtype: int64

In [7]:
#5. Verifica que exista la columna `total_venta`.
if df['total_venta'].empty:
    print('No existe')
else: 
    print('Total venta existe')

Total venta existe


In [8]:
#6. Verifica que `total_venta` coincida con `cantidad * precio_unitario`.
df["total_calculado"] = df["cantidad"] * df["precio_unitario"]

df[df['total_venta'] != df["total_calculado"]]

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,total_venta,total_calculado


#7. Escribe una observacion breve sobre el estado del dataset.
El dataset se encuentra limpio y sin nulos, el total venta coincide con la cantidad por el precio_unitario

## Parte 3. Variable Objetivo

In [9]:
#1. Puedes usar una funcion normal o una lambda.
df["venta_alta"] = df["total_venta"].apply(lambda x: 1 if x >= 1000 else 0)

In [10]:
#2. Cuenta cuantas ventas quedaron como 1 y cuantas como 0.
df["venta_alta"].value_counts()

venta_alta
1    40
0    20
Name: count, dtype: int64

Por que venta_alta es la variable objetivo?
Porque es la variable a predecir

## Parte 4. Variables De Entrada

In [11]:
#1. Crea `X`.
X = df[["cantidad", "precio_unitario", "categoria", "metodo_pago", "ciudad"]]

In [12]:
#2. Crea `y`.
y = df["venta_alta"]

In [13]:
#3. Convierte variables categoricas con `pd.get_dummies()`.
X = pd.get_dummies(X)

In [14]:
#4. Guarda la lista de columnas generadas.
columnas_modelo = X.columns.tolist()

In [15]:
#5. Muestra las primeras filas de `X` despues de `get_dummies()`.
X.head()

,cantidad,precio_unitario,categoria_Accesorios,categoria_Electronica,categoria_Muebles,metodo_pago_Efectivo,metodo_pago_Tarjeta,metodo_pago_Transferencia,ciudad_Cuernavaca,ciudad_Emiliano Zapata,ciudad_Jiutepec,ciudad_Temixco
0,1,250.0,True,False,False,True,False,False,True,False,False,False
1,1,650.0,True,False,False,False,True,False,False,False,True,False
2,1,900.0,True,False,False,False,True,False,False,False,False,True
3,1,800.0,True,False,False,True,False,False,True,False,False,False
4,2,180.0,True,False,False,True,False,False,False,False,True,False


Por que no se debe usar total_venta como variable de entrada si venta_alta se creo a partir de total_venta?
Porque total venta tiene la respuesta a la predicción que intentamos hacer, si se la damos el modelo no estaria prediciendo nada

## Parte 5. Entrenamiento Y Evaluacion

In [16]:
#1. Divide los datos en entrenamiento y prueba.
#2. Usa 80% entrenamiento y 20% prueba.
#3. Usa `random_state=42`.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [17]:
#4. Entrena el modelo.
modelo = DecisionTreeClassifier(random_state=42)
modelo.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [18]:
#5. Genera predicciones con los datos de prueba.
predicciones = modelo.predict(X_test)

In [19]:
#6. Calcula exactitud.
exactitud = accuracy_score(y_test, predicciones)
print("Exactitud:", exactitud)

Exactitud: 1.0


In [20]:
#7. Muestra matriz de confusion.
matriz = confusion_matrix(y_test, predicciones)
print(matriz)

[[4 0]
 [0 8]]


In [21]:
#8. Crea una tabla llamada `resultados_prueba` con:
resultados_prueba = pd.DataFrame({
    "valor_real": y_test,
    "prediccion": predicciones
})
resultados_prueba["coincide"] = resultados_prueba["valor_real"] == resultados_prueba["prediccion"]

In [22]:
#9. Cuenta cuantos aciertos y cuantos errores hubo.
aciertos = resultados_prueba[resultados_prueba["coincide"] == True]
errores = resultados_prueba[resultados_prueba["coincide"] == False]

print('Aciertos:', aciertos.count())

print('Errores:', errores.count())

Aciertos: valor_real    12
prediccion    12
coincide      12
dtype: int64
Errores: valor_real    0
prediccion    0
coincide      0
dtype: int64


1. Cual fue la exactitud?
La exactitud fue del 100%
2. Cuantos aciertos tuvo el modelo?
Tuvo 12 aciertos
3. Cuantos errores tuvo el modelo?
Tuvo 0 erroes
4. Que indica la matriz de confusion?
que hubo 4 coincidencias correctas y 8 coincidencias no correctas correctas
5. Una buena exactitud significa que el modelo ya es perfecto? Explica.
No, porque al agregar más datos el modelo podria fallar

## Parte 6. Guardar Modelo Y Columnas

In [23]:
#1. Usa `joblib`.
#2. Guarda el modelo entrenado.
joblib.dump(modelo, "modelo_examen_venta_alta.pkl")
#3. Guarda la lista de columnas usadas durante el entrenamiento.
joblib.dump(columnas_modelo, "columnas_examen_modelo.pkl")
#4. Verifica que los archivos aparezcan en tu carpeta.
#Si estan

['columnas_examen_modelo.pkl']

1. Para que sirve guardar el modelo?
Sirve para poder usarlo en otro DataFrame
2. Para que sirve guardar las columnas del entrenamiento?
Para que al cargar el modelo en otro DataFrame, se verifique si existen esas columnas
3. Que problema puede aparecer si no guardas las columnas?
Si se usa el modelo en otro DataFrame que no tenga esas columnas, el modelo no funcionara

## Parte 8. Cargar Modelo Y Predecir

In [24]:
#1. Carga `modelo_examen_venta_alta.pkl`.
modelo_cargado = joblib.load("modelo_examen_venta_alta.pkl")

In [25]:
#2. Carga `columnas_examen_modelo.pkl`.
columnas_modelo = joblib.load("columnas_examen_modelo.pkl")

In [26]:
#3. Carga `examen_ventas_nuevas.csv`.
nuevas = pd.read_csv("examen_ventas_nuevas.csv")

In [27]:
#4. Selecciona las mismas variables de entrada usadas en entrenamiento.
X_nuevas = nuevas[["cantidad", "precio_unitario", "categoria", "metodo_pago", "ciudad"]]

In [28]:
#5. Aplica `pd.get_dummies()`.
X_nuevas = pd.get_dummies(X_nuevas)

In [29]:
#6. Alinea columnas con `reindex()`.
X_nuevas = X_nuevas.reindex(columns=columnas_modelo, fill_value=0)

In [30]:
#7. Genera predicciones.
#8. Agrega la columna:
nuevas["prediccion_venta_alta"] = modelo_cargado.predict(X_nuevas)
nuevas["interpretacion_prediccion"] = nuevas["prediccion_venta_alta"].map({
    0: "Venta no alta",
    1: "Venta alta"
})

In [31]:
nuevas.to_csv('examen_predicciones.csv')

Que podria pasar si no usas reindex antes de predecir?
reindex alinea las columnas nuevas con las columnas del entrenamiento, si esto no se hace los datos quedan disparejos y la predicción o saldria mal o fallaria

## Parte 9. Auditoria Del Modelo

In [32]:
#1. Calcula:
nuevas["total_estimado"] = nuevas["cantidad"] * nuevas["precio_unitario"]

In [33]:
#2. Crea:
nuevas["venta_alta_real_estimada"] = nuevas["total_estimado"].apply(lambda x: 1 if x >= 1000 else 0)

In [34]:
#3. Compara:
#4. Crea la columna:
nuevas["coincide"] = nuevas["prediccion_venta_alta"] == nuevas["venta_alta_real_estimada"]
nuevas["coincide"].value_counts()

coincide
True     6
False    4
Name: count, dtype: int64

In [35]:
#5. Cuenta cuantas predicciones coincidieron y cuantas no.
#6. Filtra las ventas que no coincidieron.
errores = nuevas[nuevas["coincide"] == False]
coinciden = nuevas[nuevas["coincide"] == True]

print('Coinciden: ', coinciden.count())
print('Errores:', errores.count())

Coinciden:  id_venta                     6
fecha                        6
cliente                      6
producto                     6
categoria                    6
cantidad                     6
precio_unitario              6
metodo_pago                  6
ciudad                       6
prediccion_venta_alta        6
interpretacion_prediccion    6
total_estimado               6
venta_alta_real_estimada     6
coincide                     6
dtype: int64
Errores: id_venta                     4
fecha                        4
cliente                      4
producto                     4
categoria                    4
cantidad                     4
precio_unitario              4
metodo_pago                  4
ciudad                       4
prediccion_venta_alta        4
interpretacion_prediccion    4
total_estimado               4
venta_alta_real_estimada     4
coincide                     4
dtype: int64


In [36]:
#7. Revisa si los errores estan cerca del limite de 1000.
nuevas["distancia_a_1000"] = (nuevas["total_estimado"] - 1000).abs()
cerca_limite = nuevas[nuevas["distancia_a_1000"] <= 200]
cerca_limite

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,prediccion_venta_alta,interpretacion_prediccion,total_estimado,venta_alta_real_estimada,coincide,distancia_a_1000
0,2001,07/08/2026,Cliente A,Mouse,Accesorios,2,500,Efectivo,Cuernavaca,0,Venta no alta,1000,1,False,0
2,2003,09/08/2026,Cliente C,Mouse Pad,Accesorios,3,400,Efectivo,Temixco,0,Venta no alta,1200,1,False,200


In [37]:
#8. Guarda de nuevo `examen_predicciones.csv` con las columnas de auditoria.
nuevas.to_csv("examen_predicciones.csv", index=False)

In [38]:
nuevas

,id_venta,fecha,cliente,producto,categoria,cantidad,precio_unitario,metodo_pago,ciudad,prediccion_venta_alta,interpretacion_prediccion,total_estimado,venta_alta_real_estimada,coincide,distancia_a_1000
0,2001,07/08/2026,Cliente A,Mouse,Accesorios,2,500,Efectivo,Cuernavaca,0,Venta no alta,1000,1,False,0
1,2002,08/08/2026,Cliente B,Laptop,Electronica,1,12000,Tarjeta,Jiutepec,1,Venta alta,12000,1,True,11000
2,2003,09/08/2026,Cliente C,Mouse Pad,Accesorios,3,400,Efectivo,Temixco,0,Venta no alta,1200,1,False,200
3,2004,10/08/2026,Cliente D,Cargador,Accesorios,1,80,Tarjeta,Cuernavaca,0,Venta no alta,80,0,True,920
4,2005,11/08/2026,Cliente F,Funda Tablet,Accesorios,2,150,Efectivo,Jiutepec,0,Venta no alta,300,0,True,700
5,2006,12/08/2026,Cliente G,Soporte Celular,Accesorios,2,200,Efectivo,Emiliano Zapata,0,Venta no alta,400,0,True,600
6,2007,13/08/2026,Cliente H,Audifonos,Electronica,2,800,Tarjeta,Temixco,1,Venta alta,1600,1,True,600
7,2008,14/08/2026,Cliente I,Webcam,Electronica,1,400,Efectivo,Cuautla,1,Venta alta,400,0,False,600
8,2009,15/08/2026,Cliente J,Cable HDMI,Cables,2,120,Efectivo,Jiutepec,1,Venta alta,240,0,False,760
9,2010,16/08/2026,Cliente K,Memoria USB,Accesorios,2,80,Tarjeta,Temixco,0,Venta no alta,160,0,True,840


1. Cuantas ventas nuevas evaluaste?
10
2. Cuantas fueron predichas como venta alta?
4
3. Cuantas fueron predichas como venta no alta?
6
4. Cuantas coincidieron con la regla manual?
6
5. Cuantas no coincidieron?
4
6. Que ventas no coincidieron?
Las ventas con id: 0, 2, 7 y 8
7. Los errores estuvieron cerca del limite de 1000?
Las ventas con id: 0 y 2
8. Que paso con la categoria nueva?
Aparecio
9. Que paso con la ciudad nueva?
Aparecio